# Impedance ML Engineering — Session 11
## Build a CNN for directed impedance matrices
**Core: 60 minutes · Neural networks · Optional extension: 20–40 minutes**

## Why this matters
Does preserving local contact-pair structure help compared with summaries or flattening?

Build the judgment to implement, review, and challenge data and modeling workflows in an ML engineering role.

## Learning objectives
- Build value and validity channels without treating missing cells as real zeros.
- Predict convolution shapes and receptive fields.
- Train a small CNN using patient-separated inputs.
- Challenge assumptions behind image-style matrix models.

## R-to-Python / prerequisite bridge
PyTorch Conv2d expects (batch,channels,height,width); channel here means input feature plane, not electrode contact.

## Suggested pacing
Tasks 1–3: 18 min · Tasks 4–6: 23 min · Tasks 7–8: 11 min · Retrieval/reasoning/reflection: 8 min. Task timings are approximate; if a foundational operation needs repetition, pause there and continue later. Extensions are outside the core hour.

## Inputs and analysis contract
Start Jupyter from the repository root and select **Python (learning-python)**. Read `data/ci_telemetry/README.md` for the complete dictionary. All inputs are local, fictional teaching data: 96 patients, one implant each, 360 visits, 144 directed cells per visit. Matrix rows stimulate; columns record; contact indices are 1–12. `voltage_mv / stim_current_ua` is a synthetic kΩ V/I proxy, not a confirmed manufacturer clinical impedance algorithm. QC-valid means `ok` AND finite voltage. Mask invalid cells; unknown does not mean measured zero. No hearing, disease, or device-failure labels are available.

Write explanations in new markdown cells or print them. Exercise cells are intentionally empty. No downloads or GPU are required.

These later lessons are independently runnable: input scaffolding supplies raw representations and fixed patient indices, not model fits or engineered features. Put this setup in your first exercise cell, then add the requested code:

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from agent.course_data import load_problem, patient_partitions, NUMERIC, CATEGORICAL
problem = load_problem()
part = patient_partitions(problem)
pairs = problem["pairs"]
X = problem["X"]
y = problem["y"]
groups = problem["groups"]
matrices = problem["matrices"]   # raw current values, NaN where invalid
mask = problem["mask"]          # True where observed and usable
train, val, test = part["train"], part["val"], part["test"]
```

`train`, `val`, `test` are positions in the full prediction table. Patient counts are 54/18/24; pair counts 146/51/67. Keep IDs outside features. `target_*` fields and exact future timing are audit/outcome fields. Learn preprocessing only from the applicable training portion. Test outcomes stay reserved until Session 20. For a development-CV lesson, combine train+val, then fit all transformations inside each fold. Earlier validation is repeatedly used for learning; it is exploratory, not independent evidence.

## Concept and API reminders
A convolution applies local shared filters over the contact-pair grid. This is an inductive assumption: nearby positions have reusable structure. It is plausible only insofar as contact order and directed measurement geometry support it. The special diagonal and physical boundaries can challenge translation-style weight sharing.

For each axis: output = floor((input + 2×padding − dilation×(kernel−1) − 1)/stride + 1). A 3×3 stride-1 filter sees a local neighborhood; stacked layers/pooling enlarge the receptive field. With four output filters, two input channels, and 3×3 kernels, parameters are 4×2×3×3 plus four biases.

Imports: `import torch`, `from torch import nn`. Use `nn.Conv2d`, `nn.ReLU`, `nn.AvgPool2d`, `nn.Flatten`, and `nn.Linear`. Value/mask channels preserve the difference between a centered fill value and an observed cell. Random image augmentations require measurement-specific justification.

## Task 1 — Prepare two input channels (8 min)
Use raw current matrices and validity masks from the input scaffold. On training observed cells only calculate one global mean and SD; standardize observed values. Fill masked standardized values with 0 only AFTER retaining a mask channel (1 valid, 0 invalid). Stack to (N, 2, 12, 12). Apply the frozen training statistics to validation. Explain why filled 0 represents a centered placeholder, not measured zero impedance.

## Task 2 — Predict convolution shapes (7 min)
Create Conv2d(2, 4,kernel_size=3,padding=1), ReLU, AvgPool2d(2). Predict output shapes for a batch of 8: (8, 4, 12, 12) then (8, 4, 6, 6). Calculate convolution parameters including bias. Check with one forward pass. Explain why padding is a modeling choice around physical array boundaries.

## Task 3 — Define the small CNN (7 min)
Build two blocks: Conv2d(2, 4, 3,padding=1)→ReLU→AvgPool2d(2), then Conv2d(4, 8, 3,padding=1)→ReLU→AvgPool2d(2), then Flatten→Linear(72, 1). Predict the final spatial shape (8, 3, 3) and parameter count. No ImageNet download or RGB conversion is needed.

## Task 4 — Reason about geometry (6 min)
Draw which cell pairs a 3×3 filter combines. Explain local contact-order assumptions, the special diagonal, directed asymmetry, and what weight sharing implies. Audit the proposed augmentation `randomly rotate matrices90°`: identify which stimulation/recording meanings or anatomical conventions it changes. Keep rotations/flips disabled unless justified scientifically.

## Task 5 — Train with the established loop (10 min)
Use Adam(lr=0.01), MSELoss, batch_size 32, seed 42, max 60 epochs, validation MAE early stopping patience 8. Reuse your audited loop, adapt only input shapes and model. If this is your first neural session, read Sessions 08–10 reminders. Capture the best state by deep copy and restore it.

## Task 6 — Compare against flattened input (8 min)
Train a compact flattened-matrix MLP with width 16 using the SAME standardized value/mask inputs, train/validation partition, maximum epochs, and stopping rule. Report persistence, flattened-MLP, and CNN MAE plus parameter counts. Summary ridge is also a useful reference but uses less spatial input information.

## Task 7 — Inspect errors and channel reliance (6 min)
Plot residuals by horizon and coverage. For the frozen CNN replace validation mask channels with all 1 as a sensitivity experiment and report prediction change; label this a perturbation outside the trained input distribution. Do not interpret the result as causal importance or a new performance estimate.

## Task 8 — Write the CNN verdict (0–2 min)
State whether the architecture added validation value, its inductive assumptions, and why 144 cells still represent one visit rather than 144 labeled examples. Recommend a next experiment even if the CNN loses.

## Retrieval practice
Predict the shape after Conv2d(2, 6, 3,padding=0) on 12×12 input and explain the change from padding 1.

## Professional reasoning
1. What could go wrong if matrices are re-sorted independently per visit?
2. Why does an image pretrained model have questionable transfer assumptions here?
3. How could masking encode protocol or missingness patterns?

## Self-check and reflection
Record confidence **1–5**, hardest point, one thing now explainable, what needs repetition, and workload (too light / about right / too heavy). Restart the kernel and run your attempted cells in order. Ask for review when ready; generation does not count as completion.

## Optional extension — outside the core hour
Compare a 1D CNN over the diagonal/profile with the 2D CNN, keeping group allocation and comparable budgets. Discuss which physical geometry each assumes.

## API references
[PyTorch training basics](https://docs.pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html) · [Conv2d reference](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)